# Análise de Sentimento em Texto

Referência
- [Hugging Face: AiresPucrs/sentiment-analysis-pt](https://huggingface.co/datasets/AiresPucrs/sentiment-analysis-pt/tree/refs%2Fconvert%2Fparquet/default)

## Imports

In [ ]:
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

## Carregar Dataset

In [ ]:
data = load_dataset(path="AiresPucrs/sentiment-analysis-pt", split="train")

## Sobre os Dados

### Normalizar

In [ ]:
TEXT_COL = "text"
LABEL_COL = "label"

SENTIMENT_MAP = {
    0: "negativo",
    1: "positivo"
}


def normalize(sample):
    num_label = sample[LABEL_COL]
    text_label = SENTIMENT_MAP[num_label]

    return {
        "text": sample[TEXT_COL],
        "label": text_label
    }

In [ ]:
data = data.map(normalize)

### Separar Amostra

In [ ]:
sample = data.shuffle(seed=42).select(range(min(200, len(data))))
dataset = sample.train_test_split(test_size=0.2, seed=42)

## Tokenizador

In [ ]:
LANGUAGE_MODEL = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(LANGUAGE_MODEL)

## Sobre o Modelo

### Montar

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(LANGUAGE_MODEL)

### Pré-processamento

In [ ]:
INITIAL_PROMPT = "Classifique o sentimento: "


def preprocessing(texts):
    inputs = [
        INITIAL_PROMPT + text for text
        in texts['text']
    ]

    model = tokenizer(
        inputs,
        max_length=64,
        trucation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=texts['label'],
        max_length=4,
        trucation=True,
        padding="max_length"
    )

    model["labels"] = labels['input_ids']
    return model

In [ ]:
tokenized_dataset = dataset.map(
    preprocessing,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

### Alocar GPU

Apple Silicon GPU (MPS)

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)

print(f"Training device: {device}")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="models/text_classifier_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    predict_with_generate=True,
    logging_steps=1,
    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

### Treinamento

In [ ]:
trainer.train()

### Salvar Modelo Treinado

In [ ]:
trainer.save_model("models/text_classifier_model")
tokenizer.save_pretrained("models/text_classifier_model")

### Testar o Modelo

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("models/text_classifier_model")
model = AutoModelForSeq2SeqLM.from_pretrained("models/text_classifier_model")

In [ ]:
sentences = [
    "Adorei o curso, aprendi muito!",
    "Que decepção, esperava mais!"
]


for sentence in sentences:
    inputs = tokenizer(
        (INITIAL_PROMPT + sentence), return_tensors="pt"
    )

    output = model.generate(**inputs, max_new_tokens=4)
    predict = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Sentence: {sentence!r} -> Class: {predict}")